<font size = "7"><b> Script de Análise de dados de Eye-Tracking </b>

<font size="6"> <b>Métodos e Técnicas de Investigação III</b>

<font size = "5">Este script está <b>organizado por hipóteses</b> (hipóteses que estão definidas no relatório), sendo <b>cada capítulo o código</b> utilizado para se testar e <b>explorar cada hipótese</b>. Existe também o capítulo <b>"enviesamentos (questionário)"</b>, que se trata da <b>análise e relação</b> entre os <b>dados obtidos na experiência</b> e as possivéis <b>varíaveis confundentes</b> que identificamos na literatura e decidimos medir através de um questionário. Para além disto, existe também o capítulo <b>"Gráficos adicionais"</b>, onde foram feitos <b>dois gráficos descritivos</b> dos dados para melhor compreensão dos dados.

# <font size = "5">Hipótese 1: Em trials com estímulos de igual saliência visual, a primeira sacada ocular será predominantemente direcionada para o hemicampo esquerdo.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24",
        "pasta_l2": "2026-05-13_13-04-02_export",
        "intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},"P2": {"pasta_l1": "2026-05-06-15-38-15",
        "pasta_l2": "2026-05-13_13-18-06_export","intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export",
        "intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

largura = 1920
meio_x = largura // 2
limite_esq = meio_x - (largura * 0.10) 
limite_dir = meio_x + (largura * 0.10)

lower_green = np.array([40, 100, 50]) 
upper_green = np.array([90, 255, 255])

resultados_primeira_sacada = []

for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    caminho_video = os.path.join(caminho_base, 'Neon Scene Camera v1 ps1.mp4')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_eventos) and os.path.exists(caminho_video)): 
        continue

    cap = cv2.VideoCapture(caminho_video)
    onsets_h4 = []
    ultimo_t = -5000

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if not any(i["inicio"] <= t_ms <= i["fim"] for i in intervalos_ms): continue
        if t_ms - ultimo_t < 3500: continue 

        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), lower_green, upper_green)
        v_esq = cv2.countNonZero(mask[:, :meio_x])
        v_dir = cv2.countNonZero(mask[:, meio_x:])
        
        condicao = None
        if v_esq > 300 and v_dir > 300: 
            condicao = 'Duplo_Verde'
        elif v_esq < 50 and v_dir < 50: 
            condicao = 'Duplo_Cinzento'
            
        if condicao:
            onsets_h4.append({'ini': t_ms, 'fim': t_ms + 3000, 'condicao': condicao})
            ultimo_t = t_ms

    cap.release()

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]
    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for trial in onsets_h4:
        t_inicio_ns = inicio_gravacao_ns + (trial['ini'] * 1e6) 
        t_fim_ns = inicio_gravacao_ns + (trial['fim'] * 1e6)
        
        mask = (df_fix['start timestamp [ns]'] >= t_inicio_ns) & (df_fix['start timestamp [ns]'] <= t_fim_ns)
        fix_trial = df_fix[mask].drop_duplicates(subset=['fixation id']).sort_values('start timestamp [ns]')
        
        direcao_escolhida = None
        for _, f in fix_trial.iterrows():
            x = f['fixation x [px]']
            if x < limite_esq:
                direcao_escolhida = 'Esquerda'
                break
            elif x > limite_dir:
                direcao_escolhida = 'Direita'
                break
                
        if direcao_escolhida:
            resultados_primeira_sacada.append({
                'Participante': p_id,
                'Condicao': trial['condicao'],
                'Direcao_Primeiro_Olhar': direcao_escolhida})

df_h4 = pd.DataFrame(resultados_primeira_sacada)

if not df_h4.empty:
    contagem = df_h4['Direcao_Primeiro_Olhar'].value_counts()
    n_esq = contagem.get('Esquerda', 0)
    n_dir = contagem.get('Direita', 0)
    total = n_esq + n_dir
    
    perc_esq = (n_esq / total) * 100
    perc_dir = (n_dir / total) * 100
    
    stat, p_val = chisquare(f_obs=[n_esq, n_dir], f_exp=[total/2, total/2])

    print(f"Hemicampo Esquerdo: {perc_esq:.1f}%")
    print(f"Hemicampo Direito: {perc_dir:.1f}%")

    p_formatado = "< .001" if p_val < 0.001 else f"= {p_val:.3f}"
    print(f"Estatística: χ² = {stat:.2f}, p {p_formatado}")

    labels = ['Hemicampo Esquerdo', 'Hemicampo Direito']
    valores = [n_esq, n_dir]
    cores = ['#4C72B0', '#C44E52'] 

    fig, ax = plt.subplots(figsize=(8, 6), facecolor='white')

    wedges, texts, autotexts = ax.pie(
        valores,
        labels=labels,
        autopct='%1.1f%%',
        startangle=90,          
        colors=cores,
        pctdistance=0.75,       
        wedgeprops=dict(width=0.4, edgecolor='white', linewidth=3), 
        textprops=dict(fontsize=13, fontweight='bold', color='black'))

    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(14)

    plt.title('Direção da Primeira Sacada Ocular (Instinto Espacial)', fontweight='bold', fontsize=15, pad=20)

    plt.tight_layout()
    plt.savefig('Donut_Chart_H4_Limpo.png', dpi=300, facecolor='white', bbox_inches='tight')
    plt.show()

# <font size = "5">Hipótese 2: Os estímulos apresentados com cores salientes terão maior número de fixações comparativamente a estímulos com cores menos salientes.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export",
        "intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export",
        "intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export",
        "intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

lower_green = np.array([40, 100, 50]) 
upper_green = np.array([90, 255, 255])

resultados_fix_global = []
total_fix_verde_todos = 0
total_fix_cinza_todos = 0
total_trials_validos = 0

for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    caminho_video = os.path.join(caminho_base, 'Neon Scene Camera v1 ps1.mp4')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_video)):
        continue

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]

    cap = cv2.VideoCapture(caminho_video)
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    meio_x = largura // 2
    
    limite_esq = meio_x - (largura * 0.035)
    limite_dir = meio_x + (largura * 0.035)

    onsets_h2 = []
    ultimo_t = -5000

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if not any(i["inicio"] <= t_ms <= i["fim"] for i in intervalos_ms): continue
        if t_ms - ultimo_t < 3500: continue

        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), lower_green, upper_green)
        v_esq = cv2.countNonZero(mask[:, :meio_x])
        v_dir = cv2.countNonZero(mask[:, meio_x:])
        
        condicao = None
        if v_dir > 300 and v_esq < 50:
            condicao = 'Verde_Dir'
        elif v_esq > 300 and v_dir < 50:
            condicao = 'Verde_Esq'
            
        if condicao:
            onsets_h2.append({'ini': t_ms, 'fim': t_ms + 3000, 'condicao': condicao})
            ultimo_t = t_ms

    cap.release()
    total_trials_validos += len(onsets_h2)

    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for i, trial in enumerate(onsets_h2):
        t_inicio_ns = inicio_gravacao_ns + (trial['ini'] * 1e6) 
        t_fim_ns = inicio_gravacao_ns + (trial['fim'] * 1e6)
        
        mask = (df_fix['start timestamp [ns]'] >= t_inicio_ns) & (df_fix['start timestamp [ns]'] <= t_fim_ns)
        fix_trial = df_fix[mask].drop_duplicates(subset=['fixation id'])
        
        fix_verde_trial = 0
        fix_cinza_trial = 0
        
        for _, f in fix_trial.iterrows():
            x = f['fixation x [px]']
            
            if x < limite_esq:
                if trial['condicao'] == 'Verde_Esq': fix_verde_trial += 1
                else: fix_cinza_trial += 1
                    
            elif x > limite_dir:
                if trial['condicao'] == 'Verde_Dir': fix_verde_trial += 1
                else: fix_cinza_trial += 1
                
        total_fix_verde_todos += fix_verde_trial
        total_fix_cinza_todos += fix_cinza_trial
                
        resultados_fix_global.append({'Participante': p_id,'Trial': i + 1,'Tipo': trial['condicao'],'Bola Cinzenta': fix_cinza_trial,
            'Bola Verde': fix_verde_trial})

media_v = total_fix_verde_todos / total_trials_validos if total_trials_validos > 0 else 0
media_c = total_fix_cinza_todos / total_trials_validos if total_trials_validos > 0 else 0


print(f"TOTAL de Fixações na Bola Cinzenta: {total_fix_cinza_todos}")
print(f"TOTAL de Fixações na Bola Verde:    {total_fix_verde_todos}")

print(f"MÉDIA de Fixações por Trial (Bola Cinzenta): {media_c:.2f}")
print(f"MÉDIA de Fixações por Trial (Bola Verde):    {media_v:.2f}")
print(f"(Trials Validados Totais: {total_trials_validos})")


df_resultados_globais = pd.DataFrame(resultados_fix_global)

if not df_resultados_globais.empty:
    df_melted_global = pd.melt(df_resultados_globais, id_vars=['Participante', 'Trial', 'Tipo'], 
                               value_vars=['Bola Cinzenta', 'Bola Verde'],
                               var_name='Estímulo', value_name='Nº Fixações')

    sns.set_theme(style="whitegrid", context="talk")
    plt.figure(figsize=(10, 7))
    cores_v = ['#7f7f7f', '#2ca02c'] 
    
    sns.violinplot(data=df_melted_global, x='Estímulo', y='Nº Fixações', 
                   hue='Estímulo', palette=cores_v, inner="quart", alpha=0.4, legend=False)
                   
    sns.stripplot(data=df_melted_global, x='Estímulo', y='Nº Fixações', 
                  hue='Estímulo', palette=cores_v, size=6, jitter=0.25, 
                  alpha=0.7, edgecolor='black', linewidth=1, legend=False)
    
    plt.title('H2: Nº de Fixações por Trial (Todos os Participantes)', fontweight='bold', pad=15)
    plt.ylabel('Número de Fixações', fontweight='bold')
    sns.despine()
    plt.tight_layout()
    plt.savefig('ViolinPlot_H2_GLOBAL.png', dpi=300)
    plt.show()

<b>T-test para testar a hipótese (One-Tailed Paired samples t-test)</b> [se p<0.05 a hipótese é suportada, se não a H2 é rejeitada]

In [ ]:
import scipy.stats as stats

fixacoes_cinza = df_resultados_globais['Bola Cinzenta']
fixacoes_verde = df_resultados_globais['Bola Verde']

t_stat, p_value = stats.ttest_rel(fixacoes_verde, fixacoes_cinza, alternative='greater')

print(f"t-stat: {t_stat:.4f}")
print(f"p-value: {p_value:.5e}")


# <font size = "5">Hipótese 3:estímulos com cor apresentados à esquerda serão observados com maior intensidade do que os restantes estímulos.
></font><a class="anchor" id="python"></a>

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export",
        "intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export",
        "intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export",
        "intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

lower_green = np.array([40, 100, 50]) 
upper_green = np.array([90, 255, 255])

dados_h3 = []
total_trials = 0

for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    caminho_video = os.path.join(caminho_base, 'Neon Scene Camera v1 ps1.mp4')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_video)): continue

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]

    cap = cv2.VideoCapture(caminho_video)
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    meio_x = largura // 2
    limite_esq = meio_x - (largura * 0.035)
    limite_dir = meio_x + (largura * 0.035)

    onsets = []
    ultimo_t = -5000

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if not any(i["inicio"] <= t_ms <= i["fim"] for i in intervalos_ms): continue
        if t_ms - ultimo_t < 3500: continue

        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), lower_green, upper_green)
        v_esq = cv2.countNonZero(mask[:, :meio_x])
        v_dir = cv2.countNonZero(mask[:, meio_x:])
        
        condicao = None
        if v_dir > 300 and v_esq < 50: condicao = 'Verde_Dir'
        elif v_esq > 300 and v_dir < 50: condicao = 'Verde_Esq'
            
        if condicao:
            onsets.append({'ini': t_ms, 'fim': t_ms + 3000, 'condicao': condicao})
            ultimo_t = t_ms

    cap.release()
    total_trials += len(onsets)

    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for i, trial in enumerate(onsets):
        t_inicio_ns = inicio_gravacao_ns + (trial['ini'] * 1e6) 
        t_fim_ns = inicio_gravacao_ns + (trial['fim'] * 1e6)
        
        mask = (df_fix['start timestamp [ns]'] >= t_inicio_ns) & (df_fix['start timestamp [ns]'] <= t_fim_ns)
        fix_trial = df_fix[mask].drop_duplicates(subset=['fixation id'])
        
        tempo_esq, contagem_esq = 0, 0
        tempo_dir, contagem_dir = 0, 0
        
        for _, f in fix_trial.iterrows():
            x = f['fixation x [px]']
            duracao = f['duration [ms]']
            
            if x < limite_esq:
                tempo_esq += duracao
                contagem_esq += 1
            elif x > limite_dir:
                tempo_dir += duracao
                contagem_dir += 1

        if trial['condicao'] == 'Verde_Esq':
            dados_h3.append({'Trial_ID': f"{p_id}_{i}", 'Posicao': 'Esquerda', 'Cor': 'Verde', 'Tempo (ms)': tempo_esq, 'Nº Fixações': contagem_esq})
            dados_h3.append({'Trial_ID': f"{p_id}_{i}", 'Posicao': 'Direita', 'Cor': 'Cinzento', 'Tempo (ms)': tempo_dir, 'Nº Fixações': contagem_dir})
        elif trial['condicao'] == 'Verde_Dir':
            dados_h3.append({'Trial_ID': f"{p_id}_{i}", 'Posicao': 'Esquerda', 'Cor': 'Cinzento', 'Tempo (ms)': tempo_esq, 'Nº Fixações': contagem_esq})
            dados_h3.append({'Trial_ID': f"{p_id}_{i}", 'Posicao': 'Direita', 'Cor': 'Verde', 'Tempo (ms)': tempo_dir, 'Nº Fixações': contagem_dir})

df_h3 = pd.DataFrame(dados_h3)


medias = df_h3.groupby(['Posicao', 'Cor'])[['Tempo (ms)', 'Nº Fixações']].mean().round(2)
print(medias.to_string())



sns.set_theme(style="whitegrid", context="talk")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
paleta = {'Verde': '#2ca02c', 'Cinzento': '#7f7f7f'}

sns.barplot(data=df_h3, x='Posicao', y='Tempo (ms)', hue='Cor', palette=paleta, ax=axes[0], capsize=.1)
axes[0].set_title('Tempo de Fixação: Esquerda vs Direita', fontweight='bold')
axes[0].set_ylabel('Tempo Médio (ms)')

sns.barplot(data=df_h3, x='Posicao', y='Nº Fixações', hue='Cor', palette=paleta, ax=axes[1], capsize=.1)
axes[1].set_title('Número de Fixações: Esquerda vs Direita', fontweight='bold')
axes[1].set_ylabel('Média de Fixações')

sns.despine()
plt.tight_layout()
plt.savefig('BarPlots_H3_Global.png', dpi=300)
plt.show()

In [ ]:
import scipy.stats as stats

v_esq = df_h3[(df_h3['Cor'] == 'Verde') & (df_h3['Posicao'] == 'Esquerda')]
v_dir = df_h3[(df_h3['Cor'] == 'Verde') & (df_h3['Posicao'] == 'Direita')]
c_esq = df_h3[(df_h3['Cor'] == 'Cinzento') & (df_h3['Posicao'] == 'Esquerda')]
c_dir = df_h3[(df_h3['Cor'] == 'Cinzento') & (df_h3['Posicao'] == 'Direita')]



def testar_hipotese(metrica):
    print(f"\nMÉTRICA: {metrica.upper()}")

    t1, p1 = stats.ttest_ind(v_esq[metrica], v_dir[metrica], alternative='greater')
    print(f"vs Verde Direita    -> t: {t1:.3f} | p-value: {p1:.4f}")

    t2, p2 = stats.ttest_ind(v_esq[metrica], c_esq[metrica], alternative='greater')
    print(f"vs Cinza Esquerda   -> t: {t2:.3f} | p-value: {p2:.4f}")

    df_paired = df_h3[df_h3['Trial_ID'].isin(v_esq['Trial_ID'])]
    t3, p3 = stats.ttest_rel(df_paired[(df_paired['Cor'] == 'Verde')]['Tempo (ms)' if metrica == 'Tempo (ms)' else 'Nº Fixações'], 
        df_paired[(df_paired['Cor'] == 'Cinzento')]['Tempo (ms)' if metrica == 'Tempo (ms)' else 'Nº Fixações'], 
        alternative='greater')
    print(f"vs Cinza Direita    -> t: {t3:.3f} | p-value: {p3:.4f}")

testar_hipotese('Tempo (ms)')
testar_hipotese('Nº Fixações')

<b>Tabela dos t-tests (feito em HTML)</b>

In [ ]:
%%html
<div style="background-color: white; padding: 20px; display: flex; justify-content: left;">
    <table style="border-collapse: collapse; width: 680px; font-family: 'Times New Roman', Times, serif; font-size: 12pt; text-align: left; color: black; background-color: white;">
      <caption style="text-align: left; margin-bottom: 10px; color: black;">
        <strong>Tabela 2</strong><br>
        <em>Resultados dos One-Tailed t-Tests para as Comparações Cruzadas da Hipótese 3</em>
      </caption>
      <thead>
        <tr style="border-top: 2px solid black; border-bottom: 1px solid black; background-color: white;">
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white;">Métrica de Atenção</th>
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white;">Comparação (Verde à Esquerda vs.)</th>
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white; text-align: center;"><em>t</em></th>
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white; text-align: center;"><em>p</em></th>
        </tr>
      </thead>
      <tbody style="background-color: white;">
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"><strong>Tempo de Fixação (ms)</strong></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Verde à Direita</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">-2.24</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">.986</td>
        </tr>
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Cinzento à Esquerda</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">2.26</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;"><strong>.014*</strong></td>
        </tr>
        <tr style="background-color: white; border-bottom: 1px solid #ddd;">
          <td style="padding: 8px 5px; color: black; background-color: white;"></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Cinzento à Direita</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">-0.86</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">.801</td>
        </tr>
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"><strong>Número de Fixações</strong></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Verde à Direita</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">-1.54</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">.936</td>
        </tr>
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Cinzento à Esquerda</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">0.40</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">.346</td>
        </tr>
        <tr style="border-bottom: 2px solid black; background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Cinzento à Direita</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">0.36</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">.361</td>
        </tr>
      </tbody>
      <tfoot style="background-color: white;">
        <tr style="background-color: white;">
          <td colspan="4" style="padding: 8px 5px; font-size: 11pt; color: black; background-color: white; text-align: left;">
            <em>Nota.</em> Análise efetuada com base em <em>N</em> = 60 trials validados. * <em>p</em> < .05.
          </td>
        </tr>
      </tfoot>
    </table>
</div>

<b>Tabela das médias (feito em HTML)</b>

In [ ]:
%%html
<div style="background-color: white; padding: 20px; display: flex; justify-content: left;">
    <table style="border-collapse: collapse; width: 680px; font-family: 'Times New Roman', Times, serif; font-size: 12pt; text-align: left; color: black; background-color: white;">
      <caption style="text-align: left; margin-bottom: 10px; color: black;">
        <strong>Tabela 1</strong><br>
        <em>Médias do Tempo Total e Número de Fixações em Função da Posição Espacial e Saliência da Cor</em>
      </caption>
      <thead>
        <tr style="border-top: 2px solid black; border-bottom: 1px solid black; background-color: white;">
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white;">Posição Espacial</th>
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white;">Cor do Estímulo</th>
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white; text-align: center;">Tempo Médio de Fixação (ms)</th>
          <th style="padding: 10px 5px; font-weight: normal; color: black; background-color: white; text-align: center;">Média de Fixações</th>
        </tr>
      </thead>
      <tbody style="background-color: white;">
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"><strong>Esquerda</strong></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Verde (Saliente)</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">738.75</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">0.52</td>
        </tr>
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Cinzento (Neutro)</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">209.87</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">0.42</td>
        </tr>
        <tr style="background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"><strong>Direita</strong></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Verde (Saliente)</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">2487.32</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">0.87</td>
        </tr>
        <tr style="border-bottom: 2px solid black; background-color: white;">
          <td style="padding: 8px 5px; color: black; background-color: white;"></td>
          <td style="padding: 8px 5px; color: black; background-color: white;">Cinzento (Neutro)</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">1181.58</td>
          <td style="padding: 8px 5px; color: black; background-color: white; text-align: center;">0.45</td>
        </tr>
      </tbody>
      <tfoot style="background-color: white;">
        <tr style="background-color: white;">
          <td colspan="4" style="padding: 8px 5px; font-size: 11pt; color: black; background-color: white; text-align: left;">
            <em>Nota.</em> O número total de trials validados para esta análise foi de <em>N</em> = 60.
          </td>
        </tr>
      </tfoot>
    </table>
</div>

# <font size = "5">Hipótese 4: O tempo total de fixação será semelhante entre os estímulos com cores salientes apresentados à direita e estímulos com cores menos salientes apresentados pelo lado esquerdo.<font color='Blue'></font><a class="anchor" id="python"></a>

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export",
        "intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export",
        "intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export",
        "intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

lower_green = np.array([40, 100, 50]) 
upper_green = np.array([90, 255, 255])

resultados_tempo_global = []
dados_heatmap_global = []
bg_image_global = None 

tempo_total_esq_todos = 0
tempo_total_dir_todos = 0
total_trials_validos = 0

for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    caminho_video = os.path.join(caminho_base, 'Neon Scene Camera v1 ps1.mp4')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_video)):
        continue

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]

    cap = cv2.VideoCapture(caminho_video)
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    meio_x = largura // 2
    
    limite_esq = meio_x - (largura * 0.035)
    limite_dir = meio_x + (largura * 0.035)

    onsets_h4 = []
    ultimo_t = -5000

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if not any(i["inicio"] <= t_ms <= i["fim"] for i in intervalos_ms): continue
        if t_ms - ultimo_t < 3500: continue

        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), lower_green, upper_green)
        v_esq = cv2.countNonZero(mask[:, :meio_x])
        v_dir = cv2.countNonZero(mask[:, meio_x:])
        
        if v_dir > 300 and v_esq < 50:
            onsets_h4.append({'ini': t_ms, 'fim': t_ms + 3000})
            ultimo_t = t_ms

    total_trials_validos += len(onsets_h4)

    if bg_image_global is None and len(onsets_h4) > 0:
        cap.set(cv2.CAP_PROP_POS_MSEC, onsets_h4[0]['ini'] + 1500)
        ret, frame_bg = cap.read()
        if ret:
            bg_image_global = cv2.cvtColor(frame_bg, cv2.COLOR_BGR2RGB)
    
    cap.release()

    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for i, trial in enumerate(onsets_h4):
        t_inicio_ns = inicio_gravacao_ns + (trial['ini'] * 1e6) 
        t_fim_ns = inicio_gravacao_ns + (trial['fim'] * 1e6)
        
        mask = (df_fix['start timestamp [ns]'] >= t_inicio_ns) & (df_fix['start timestamp [ns]'] <= t_fim_ns)
        fix_trial = df_fix[mask].drop_duplicates(subset=['fixation id'])
        
        duracao_esq_trial = 0
        duracao_dir_trial = 0
        
        for _, f in fix_trial.iterrows():
            x = f['fixation x [px]']
            duracao = f['duration [ms]']
            y = f.get('fixation y [px]', altura / 2)
                
            if x < limite_esq:
                duracao_esq_trial += duracao
                tempo_total_esq_todos += duracao
                dados_heatmap_global.append({'x': x, 'y': y, 'peso': duracao})
            elif x > limite_dir:
                duracao_dir_trial += duracao
                tempo_total_dir_todos += duracao
                dados_heatmap_global.append({'x': x, 'y': y, 'peso': duracao})
                
        resultados_tempo_global.append({'Participante': p_id,'Trial': i + 1,
            'Cinzenta (Esquerda)': duracao_esq_trial,'Verde (Direita)': duracao_dir_trial})

media_esq = tempo_total_esq_todos / total_trials_validos if total_trials_validos > 0 else 0
media_dir = tempo_total_dir_todos / total_trials_validos if total_trials_validos > 0 else 0

print(f"Média Cinzenta (Esq): {media_esq:.0f} ms | Média Verde (Dir): {media_dir:.0f} ms")

df_resultados_globais = pd.DataFrame(resultados_tempo_global)

if not df_resultados_globais.empty:
    df_melted_global = pd.melt(df_resultados_globais, id_vars=['Participante', 'Trial'], value_vars=['Cinzenta (Esquerda)', 'Verde (Direita)'],
                               var_name='Lado (Estímulo)', value_name='Tempo (ms)')

    sns.set_theme(style="whitegrid", context="talk")
    plt.figure(figsize=(10, 7))
    cores_v = ['#7f7f7f', '#2ca02c'] 
    
    sns.violinplot(data=df_melted_global, x='Lado (Estímulo)', y='Tempo (ms)', hue='Lado (Estímulo)', palette=cores_v, inner="quart", alpha=0.4, legend=False)
                   
    sns.stripplot(data=df_melted_global, x='Lado (Estímulo)', y='Tempo (ms)', hue='Lado (Estímulo)', palette=cores_v, size=6, jitter=0.25, 
                  alpha=0.7, edgecolor='black', linewidth=1, legend=False)
    
    plt.title('H4: Tempo de Fixação por Trial (Global)', fontweight='bold', pad=15)
    plt.ylabel('Tempo de Fixação (ms)', fontweight='bold')
    sns.despine()
    plt.tight_layout()
    plt.savefig('ViolinPlot_H4_GLOBAL.png', dpi=300)
    plt.show()

df_heat_global = pd.DataFrame(dados_heatmap_global)

if not df_heat_global.empty and bg_image_global is not None:
    plt.figure(figsize=(12, 12 * (bg_image_global.shape[0] / bg_image_global.shape[1])))
    plt.imshow(bg_image_global, extent=[0, largura, altura, 0])
    
    sns.kdeplot(data=df_heat_global, x='x', y='y', weights='peso',fill=True, cmap='turbo', alpha=0.7, thresh=0.001,  levels=100,bw_adjust=0.5)
    
    plt.xlim(0, largura)
    plt.ylim(altura, 0)
    plt.axis('off')
    plt.title('H4: Heatmap Global de Fixações', color='white', backgroundcolor='black', fontweight='bold', fontsize=14)
    plt.savefig('Heatmap_H4_GLOBAL.png', dpi=300, bbox_inches='tight', pad_inches=0)
    plt.show()

<b>T-test para testar a hipótese (Paired samples t-test)</b> [se p>0.05 a hipótese é suportada, se não a H4 é rejeitada]

In [ ]:
import scipy.stats as stats

tempos_esq = df_resultados_globais['Cinzenta (Esquerda)']
tempos_dir = df_resultados_globais['Verde (Direita)']

t_stat, p_value = stats.ttest_rel(tempos_dir, tempos_esq)

print(f"t-stat: {t_stat:.4f}")
print(f"p-value: {p_value:.5f}")

# <font size = "5">Enviesamentos (questionário)<font color='Blue'></font><a class="anchor" id="python"></a>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if 'df_h4' in locals() and not df_h4.empty:

    tabela_cruzada = pd.crosstab(df_h4['Participante'], df_h4['Direcao_Primeiro_Olhar'])
    print(tabela_cruzada)
    print("\n")

    sns.set_theme(style="whitegrid", context="talk")
    plt.figure(figsize=(10, 6))

    ax = sns.countplot(data=df_h4, x='Participante', hue='Direcao_Primeiro_Olhar', palette={'Esquerda': '#4C72B0', 'Direita': '#C44E52'})
    
    ax.set_title('Direção da Primeira Sacada por Participante', fontweight='bold', pad=15)
    ax.set_ylabel('Número de Trials')
    ax.set_xlabel('Participante')
    plt.legend(title='Hemicampo')

    for p in ax.patches:
        altura = p.get_height()
        if pd.notnull(altura) and altura > 0:
            ax.annotate(f'{int(altura)}', (p.get_x() + p.get_width() / 2., altura),ha='center', va='center', fontsize=12, color='black', xytext=(0, 8), textcoords='offset points')

    plt.tight_layout()
    plt.savefig('H4_Por_Participante.png', dpi=300)
    plt.show()

else:
    print("Erro")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export","intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export","intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export","intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

largura = 1920 
meio_x = largura // 2

limite_esq = meio_x - (largura * 0.10) 
limite_dir = meio_x + (largura * 0.10)

dados_inercia = []


for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_eventos)): 
        continue

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]

    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    tempo_total_ms = 0
    tempo_centro_ms = 0

    for _, f in df_fix.iterrows():
        t_fix_ns = f['start timestamp [ns]']
        t_fix_ms = (t_fix_ns - inicio_gravacao_ns) / 1e6
        duracao = f['duration [ms]']

        if any(i["inicio"] <= t_fix_ms <= i["fim"] for i in intervalos_ms):
            tempo_total_ms += duracao
            x = f['fixation x [px]']

            if limite_esq <= x <= limite_dir:
                tempo_centro_ms += duracao
                
    if tempo_total_ms > 0:
        perc_centro = (tempo_centro_ms / tempo_total_ms) * 100
        dados_inercia.append({'Participante': p_id,'Percentagem_Inercia': perc_centro,'Perfil': 'Visão Não Corrigida' if p_id == 'P1' else ('Neurodivergente' if p_id == 'P2' else 'Normativo')})

df_inercia = pd.DataFrame(dados_inercia)

sns.set_theme(style="whitegrid", context="talk")
plt.figure(figsize=(9, 6))

ax = sns.barplot(data=df_inercia, x='Participante', y='Percentagem_Inercia', hue='Perfil',dodge=False,palette=['#C44E52', '#E6842A', '#4C72B0'] )

ax.set_title('Taxa de Inércia Central por Participante', fontweight='bold', pad=15)
ax.set_ylabel('Tempo no Centro do Ecrã (%)')
ax.set_xlabel('Participante')
ax.set_ylim(0, 100) 

for p in ax.patches:
    altura = p.get_height()
    if pd.notnull(altura) and altura > 0:
        ax.annotate(f'{altura:.1f}%', (p.get_x() + p.get_width() / 2., altura),ha='center', va='center', fontsize=14, color='black', fontweight='bold',xytext=(0, 10), textcoords='offset points')

plt.legend(title='Perfil Clínico', loc='upper right')
plt.tight_layout()
plt.savefig('Comprovacao_Inercia_Central.png', dpi=300)
plt.show()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24", "pasta_l2": "2026-05-13_13-04-02_export", "intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}], "dominancia": "Destros (P1 e P3)"},
    "P2": {"pasta_l1": "2026-05-06-15-38-15", "pasta_l2": "2026-05-13_13-18-06_export", "intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}], "dominancia": "Canhotos (P2)"},
    "P3": {"pasta_l1": "2026-05-06-15-54-31", "pasta_l2": "2026-05-13_13-28-24_export", "intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}], "dominancia": "Destros (P1 e P3)"}}

largura = 1920
meio_x = largura // 2
limite_esq = meio_x - (largura * 0.10) 
limite_dir = meio_x + (largura * 0.10)

lower_green = np.array([40, 100, 50]) 
upper_green = np.array([90, 255, 255])

resultados_direcao = []

for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    caminho_video = os.path.join(caminho_base, 'Neon Scene Camera v1 ps1.mp4')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_video)): continue

    cap = cv2.VideoCapture(caminho_video)
    onsets = []
    ultimo_t = -5000

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if not any(i["inicio"] <= t_ms <= i["fim"] for i in intervalos_ms): continue
        if t_ms - ultimo_t < 3500: continue 

        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), lower_green, upper_green)
        v_esq = cv2.countNonZero(mask[:, :meio_x])
        v_dir = cv2.countNonZero(mask[:, meio_x:])

        if (v_esq > 300 and v_dir > 300) or (v_esq < 50 and v_dir < 50):
            onsets.append({'ini': t_ms, 'fim': t_ms + 3000})
            ultimo_t = t_ms
    cap.release()

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]
    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for trial in onsets:
        t_inicio_ns = inicio_gravacao_ns + (trial['ini'] * 1e6) 
        t_fim_ns = inicio_gravacao_ns + (trial['fim'] * 1e6)
        mask = (df_fix['start timestamp [ns]'] >= t_inicio_ns) & (df_fix['start timestamp [ns]'] <= t_fim_ns)
        fix_trial = df_fix[mask].drop_duplicates(subset=['fixation id']).sort_values('start timestamp [ns]')
        
        for _, f in fix_trial.iterrows():
            x = f['fixation x [px]']
            if x < limite_esq:
                resultados_direcao.append({'Dominancia': p_dados['dominancia'], 'Direcao': 'Esquerda'})
                break
            elif x > limite_dir:
                resultados_direcao.append({'Dominancia': p_dados['dominancia'], 'Direcao': 'Direita'})
                break

df_dir = pd.DataFrame(resultados_direcao)

if not df_dir.empty:
    cruzamento = pd.crosstab(df_dir['Dominancia'], df_dir['Direcao'], normalize='index') * 100
    

    print(cruzamento.round(1).astype(str) + '%')

    sns.set_theme(style="white", context="talk")
    fig, ax = plt.subplots(figsize=(8, 6))

    cruzamento.plot(kind='bar', stacked=True, color=['#C44E52', '#4C72B0'], ax=ax, edgecolor='white', width=0.6)

    ax.set_title('Direção da Primeira Sacada por Dominância Manual', fontweight='bold', pad=15)
    ax.set_ylabel('Percentagem das Sacadas (%)')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontweight='bold')

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(reversed(handles), reversed(labels), title='Lado Escolhido', loc='upper right', bbox_to_anchor=(1.3, 1))

    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontweight='bold', fontsize=14)

    plt.tight_layout()
    plt.savefig('Comprovacao_Direcao_Dominancia.png', dpi=300)
    plt.show()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export","intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}],"audicao": "Défice Ouvido Dir. (P1)"},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export","intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}],"audicao": "Audição Normal (P2, P3)"},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export","intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}],"audicao": "Audição Normal (P2, P3)"}}

largura = 1920
meio_x = largura // 2
limite_esq = meio_x - (largura * 0.10) 
limite_dir = meio_x + (largura * 0.10)

lower_green = np.array([40, 100, 50]) 
upper_green = np.array([90, 255, 255])

contagem_olhares = []


for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    caminho_video = os.path.join(caminho_base, 'Neon Scene Camera v1 ps1.mp4')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_video)): 
        continue

    cap = cv2.VideoCapture(caminho_video)
    onsets_neutros = []
    ultimo_t = -5000

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        t_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if not any(i["inicio"] <= t_ms <= i["fim"] for i in intervalos_ms): continue
        if t_ms - ultimo_t < 3500: continue 

        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV), lower_green, upper_green)
        v_esq = cv2.countNonZero(mask[:, :meio_x])
        v_dir = cv2.countNonZero(mask[:, meio_x:])

        if (v_esq > 300 and v_dir > 300) or (v_esq < 50 and v_dir < 50):
            onsets_neutros.append({'ini': t_ms, 'fim': t_ms + 3000})
            ultimo_t = t_ms
            
    cap.release()

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]
    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for trial in onsets_neutros:
        t_inicio_ns = inicio_gravacao_ns + (trial['ini'] * 1e6) 
        t_fim_ns = inicio_gravacao_ns + (trial['fim'] * 1e6)
        mask = (df_fix['start timestamp [ns]'] >= t_inicio_ns) & (df_fix['start timestamp [ns]'] <= t_fim_ns)
        fix_trial = df_fix[mask].drop_duplicates(subset=['fixation id'])
        
        for _, f in fix_trial.iterrows():
            x = f['fixation x [px]']
            if x < limite_esq:
                contagem_olhares.append({'Audicao': p_dados['audicao'], 'Direcao': 'Esquerda'})
            elif x > limite_dir:
                contagem_olhares.append({'Audicao': p_dados['audicao'], 'Direcao': 'Direita'})

df_olhares = pd.DataFrame(contagem_olhares)

if not df_olhares.empty:

    cruzamento = pd.crosstab(df_olhares['Audicao'], df_olhares['Direcao'], normalize='index') * 100
    

    print(cruzamento.round(1).astype(str) + '%')

    sns.set_theme(style="white", context="talk")
    fig, ax = plt.subplots(figsize=(9, 6))

    cruzamento.plot(kind='bar', stacked=True, color=['#C44E52', '#4C72B0'], ax=ax, edgecolor='white', width=0.6)

    ax.set_title('Número de Fixações em Estímulos Idênticos por Perfil Auditivo', fontweight='bold', pad=15)
    ax.set_ylabel('Proporção de Olhares (%)')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontweight='bold')
    
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(reversed(handles), reversed(labels), title='Lado Fixado', loc='upper right', bbox_to_anchor=(1.35, 1))

    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontweight='bold', fontsize=14)

    plt.tight_layout()
    plt.savefig('Comprovacao_Fixacoes_Audicao.png', dpi=300)
    plt.show()
else:
    print("Não foram encontrados olhares suficientes para processar.")

# <font size = "5">Gráficos adicionais<font color='Blue'></font><a class="anchor" id="python"></a>

<b>Heatmap geral (todos os trials)</b>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export","intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export","intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export","intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

largura = 1920 
altura = 1080

coords_gerais = {'x': [], 'y': [], 'duracao': []}


for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_eventos)): 
        continue

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]

    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for _, f in df_fix.iterrows():
        t_fix_ns = f['start timestamp [ns]']
        t_fix_ms = (t_fix_ns - inicio_gravacao_ns) / 1e6

        if any(i["inicio"] <= t_fix_ms <= i["fim"] for i in intervalos_ms):
            coords_gerais['x'].append(f['fixation x [px]'])
            coords_gerais['y'].append(f['fixation y [px]'])
            coords_gerais['duracao'].append(f['duration [ms]'])

df_geral = pd.DataFrame(coords_gerais)

sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(figsize=(14, 8))

nome_imagem_fundo = 'imagem_fundo_4.png' 

try:
    fundo = plt.imread(nome_imagem_fundo) 
    ax.imshow(fundo, extent=[0, largura, altura, 0])

except FileNotFoundError:
    print(f"Imagem de fundo não encontrada")

if not df_geral.empty:sns.kdeplot(data=df_geral, x='x', y='y', weights='duracao', cmap="turbo", fill=True, 
        bw_adjust=0.3,levels=100,alpha=0.6,ax=ax,thresh=0.001)

ax.set_xlim(0, largura)
ax.set_ylim(altura, 0)
ax.set_title('Panorama Geral: Tempo de Fixação (Todos os Trials)', fontweight='bold', pad=15)
ax.set_xlabel('Eixo X')
ax.set_ylabel('Eixo Y')

plt.tight_layout()
plt.savefig('Heatmap_Panorama_Geral_Fundo_Imagem2.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def tempo_para_ms(tempo_str):
    partes = tempo_str.split(':')
    return (int(partes[0]) * 60 + int(partes[1])) * 1000 + int(partes[2]) * 10

participantes = {"P1": {"pasta_l1": "2026-05-06-15-19-24","pasta_l2": "2026-05-13_13-04-02_export",
        "intervalos": [{"inicio": "01:07:15", "fim": "02:48:00"}, {"inicio": "02:51:19", "fim": "04:32:00"}]},
    "P2": {"pasta_l1": "2026-05-06-15-38-15","pasta_l2": "2026-05-13_13-18-06_export",
        "intervalos": [{"inicio": "00:51:03", "fim": "02:31:00"}, {"inicio": "02:35:10", "fim": "04:15:00"}]},
    "P3": {"pasta_l1": "2026-05-06-15-54-31","pasta_l2": "2026-05-13_13-28-24_export",
        "intervalos": [{"inicio": "01:18:08", "fim": "02:58:00"}, {"inicio": "03:01:00", "fim": "04:42:00"}]}}

largura = 1920 
altura = 1080

coords_gerais = {'x': [], 'y': []}

for p_id, p_dados in participantes.items():
    intervalos_ms = [{"inicio": tempo_para_ms(i["inicio"]), "fim": tempo_para_ms(i["fim"])} for i in p_dados["intervalos"]]
    
    caminho_base = os.path.join(p_dados["pasta_l1"], p_dados["pasta_l2"])
    caminho_fixacoes = os.path.join(caminho_base, 'fixations.csv')
    caminho_eventos = os.path.join(caminho_base, 'events.csv')
    
    if not (os.path.exists(caminho_fixacoes) and os.path.exists(caminho_eventos)): 
        continue

    df_fix = pd.read_csv(caminho_fixacoes)
    df_ev = pd.read_csv(caminho_eventos)
    inicio_gravacao_ns = df_ev[df_ev['name'] == 'recording.begin']['timestamp [ns]'].iloc[0]

    df_fix['start timestamp [ns]'] = df_fix['start timestamp [ns]'].astype('int64')

    for _, f in df_fix.iterrows():
        t_fix_ns = f['start timestamp [ns]']
        t_fix_ms = (t_fix_ns - inicio_gravacao_ns) / 1e6

        if any(i["inicio"] <= t_fix_ms <= i["fim"] for i in intervalos_ms):
            coords_gerais['x'].append(f['fixation x [px]'])
            coords_gerais['y'].append(f['fixation y [px]'])

df_geral = pd.DataFrame(coords_gerais)

sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(figsize=(14, 8))

nome_imagem_fundo = 'imagem_fundo_4.png' 

try:
    fundo = plt.imread(nome_imagem_fundo) 
    ax.imshow(fundo, extent=[0, largura, altura, 0])
except FileNotFoundError:
    print(f"imagem não econtrada")

if not df_geral.empty:
    sns.scatterplot(data=df_geral, x='x', y='y', color='red', alpha=0.3, s=20,edgecolor=None,ax=ax)

ax.set_xlim(0, largura)
ax.set_ylim(altura, 0) 
ax.set_title('Mapa de Fixações Globais (Todos os Participantes)', fontweight='bold', pad=15)
ax.set_xlabel('Eixo X (Pixéis)')
ax.set_ylabel('Eixo Y (Pixéis)')

plt.tight_layout()
plt.savefig('Mapa_Fixacoes_Fundo_Imagem4.png', dpi=300)
plt.show()